# Domain shift baseline

This notebook is fully self contained. It trains the Spain source domain
model if it does not already exist (skips instantly if it does, this is safe
to rerun any time), then evaluates it zero shot on the real Malawi test set,
both with and without CLAHE preprocessing, then runs the t-SNE domain shift
analysis. Nothing here needs a checkpoint to already exist somewhere else.

Reproducibility note: every training and evaluation step below reads its
seed from the experiment config, and scripts/06_train.py refuses to run
against uncommitted code, see src/fetal_ai/provenance.py. Rerunning this
notebook from the same commit produces the same checkpoints and the same
numbers, it does not retrain anything that already has a saved result.

In [ ]:
import sys, json
sys.path.insert(0, "src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image as IPImage, display


## Step 1: Spain baseline model

Trains from ImageNet weights on the Spain source domain, no African data
involved at all. Every African experiment in this project fine tunes from
this checkpoint. If results/baseline_spain_efficientnet_b0/metrics.json
already exists, this skips training entirely and the cell below just
confirms that.

In [ ]:
!python scripts/06_train.py --config configs/experiment/baseline_spain.yaml


In [ ]:
spain_result = json.load(open("results/baseline_spain_efficientnet_b0/metrics.json"))
print(f"Best validation F1 macro: {spain_result['metrics']['best_val_f1_macro']:.4f}")
print(f"Best epoch: {spain_result['metrics']['best_epoch']}")
print(f"Git commit this checkpoint was trained from: {spain_result['provenance']['git_commit'][:8]}")


## Step 2: zero shot evaluation on the real Malawi test set

The Spain checkpoint has never seen an African image. Evaluated twice
against the same 60 image, 22 patient Malawi test set, once with no
preprocessing change and once with CLAHE applied at inference time only,
matching Table 4's first two rows in the original submission. Both are
resume safe, rerunning this cell after the first successful run just
reloads the saved result.

In [ ]:
!python scripts/08_evaluate.py \
    --checkpoint results/baseline_spain_efficientnet_b0/checkpoint.pt \
    --run_id spain_zero_shot_on_malawi


In [ ]:
!python scripts/08_evaluate.py \
    --checkpoint results/baseline_spain_efficientnet_b0/checkpoint.pt \
    --run_id spain_plus_clahe_on_malawi --use_clahe


In [ ]:
zero_shot = json.load(open("results/spain_zero_shot_on_malawi/metrics.json"))
plus_clahe = json.load(open("results/spain_plus_clahe_on_malawi/metrics.json"))

def summarize(name, result):
    m = result["metrics"]
    ci = m["f1_macro_bootstrap_ci"]
    return {
        "run": name,
        "F1 macro": m["point_estimates"]["f1_macro"],
        "95% CI lower": ci["ci_lower"],
        "95% CI upper": ci["ci_upper"],
        "accuracy": m["point_estimates"]["accuracy"],
        "pr_auc_macro": m["point_estimates"].get("pr_auc_macro"),
        "n_patients": m["n_patients"],
    }

comparison = pd.DataFrame([
    summarize("zero shot", zero_shot),
    summarize("zero shot + CLAHE", plus_clahe),
]).set_index("run")
comparison


## Confusion matrices

Pulled directly from each evaluation's saved point_estimates, not
recomputed here.

In [ ]:
class_names = zero_shot["metrics"]["class_names"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, result) in zip(axes, [("Zero shot", zero_shot), ("Zero shot + CLAHE", plus_clahe)]):
    cm = np.array(result["metrics"]["point_estimates"]["confusion_matrix"])
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names)
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(name)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black")

plt.tight_layout()
plt.show()


## Step 3: domain shift analysis

t-SNE on the real penultimate layer embeddings (1280 dimensions, confirmed
directly against the model architecture, see DECISIONS_LOG.md), 200 Spain
images and 100 African images by default, matching the original paper's
Figure 3 sample sizes. Also computes a real, cross validated number for how
separable the two domains are in this representation, not just a plot to
eyeball.

In [ ]:
!python scripts/10_tsne_analysis.py


In [ ]:
display(IPImage(filename="results/tsne_domain_shift.png"))

separability = json.load(open("results/tsne_domain_shift.json"))["domain_separability"]
print(f"\nDomain separability, cross validated linear classifier accuracy on the real embeddings:")
print(f"  {separability['mean_accuracy']:.3f} +/- {separability['std_accuracy']:.3f} "
      f"(chance level = {separability['chance_level']:.3f}, n={separability['n_samples']})")

gap_above_chance = separability["mean_accuracy"] - separability["chance_level"]
if gap_above_chance > 0.15:
    print("Domains remain clearly separable in this representation.")
elif gap_above_chance > 0.05:
    print("Domains show some residual separability, not fully closed.")
else:
    print("Domains are close to indistinguishable by a linear classifier, near chance level.")


## Summary

Every number and figure above came from a real, resume safe pipeline run,
not a hand typed table. The comparison, confusion matrices, and domain
separability score together are what actually support (or do not support)
the original submission's domain shift claims, rather than a single
headline percentage.

Next notebook: pooled baseline, LOCO cross validation, and model soup,
including the finding that pooled fine tuning and LOCO plus model soup
produce identical predictions on the real Malawi test set.